# Pediatric EEG V2 — full-cohort experiment

All 23 individuals; frozen 15/4/4 split. Select **T4 GPU, runtime version 2026.07 (Python 3.12)** before starting. This notebook uses the repository implementation.

**Drive-free:** raw EDFs, int16 caches, and generated arrays stay on the Colab runtime disk. Compact recovery archives are downloaded to the gitignored local project `recovery/` folder at each completed epoch and natural checkpoint. Google Drive is never mounted or accessed.

Research/educational prototype. Not for diagnosis or clinical decisions. No test-based selection.


In [ ]:
import sys
assert sys.version_info[:2] == (3, 12), "Select runtime version 2026.07 (Python 3.12)"
print("Drive-free workflow; runtime disk only. No Drive permissions are needed.")


In [ ]:
import os, subprocess, shutil
from pathlib import Path
REPO = "https://github.com/iampenuel/pediatric-eeg-seizure-v2.git"
PROJECT = Path("/content/pediatric-eeg-seizure-v2")
def run(command):
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait():
        raise RuntimeError(f"Command failed: {command}")
if not PROJECT.exists():
    run(["git", "clone", REPO, str(PROJECT)])
os.chdir(PROJECT)
run(["git", "rev-parse", "HEAD"])
run(["nvidia-smi"])
print("Free runtime GiB:", round(shutil.disk_usage("/content").free / 2**30, 1))


In [ ]:
run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "-e", ".[data,train,export,serve,test]"])


In [ ]:
run([sys.executable, "-m", "pytest", "-q"])


## Full-cohort run with verified local recovery

The workflow prepares and audits all recordings, trains both models, freezes validation-only selection, evaluates held-out patients, and builds the real demo release.

At each recovery prompt, type `download`. Save the archive into the local project's gitignored `recovery/` directory, verify the displayed SHA-256 and all archive members with `python -m seizure_v2.recovery verify --archive ARCHIVE --sha256 SHA`, then enter that SHA at the Colab prompt. The agent can handle this through Chrome. Training waits until verification is acknowledged. Keep the latest two verified snapshots; do not accumulate every epoch locally.

After runtime loss, upload the latest compact archive into `/content` using Colab's Files pane, clone the **recorded Git commit**, install dependencies, and restore to a new output directory using `python -m seizure_v2.recovery restore --archive /content/ARCHIVE --sha256 SHA --output /content/eeg-v2-run/full-seed42`. Rerun the workflow: it rebuilds runtime caches, checks identical data/configuration, and resumes optimizer/RNG state from the completed epoch. Frozen runs remain frozen. Never change the configuration after test inspection.

The final demo archive is downloaded separately. GPU availability/runtime duration are not guaranteed; no paid resources or reduced cohort are assumed.


In [ ]:
import sys
# Newly installed editable paths are not loaded automatically by an existing kernel.
sys.path.insert(0, str(PROJECT / "src"))
DATA = "/content/chbmit-v2"
OUTPUT = "/content/eeg-v2-run/full-seed42"
from scripts.colab_runner import run_with_downloads
run_with_downloads(DATA, OUTPUT)


In [ ]:
import json
summary = json.loads((Path(OUTPUT) / "reports/summary.json").read_text())
for name, report in summary.items():
    print(name, "pooled:", report["pooled"], "patient macro:", report["patient_macro"])
print("Demo release:", Path(OUTPUT) / "demo.tar.gz")
